# Denovo docking run — input preparation & checks

Both inputs to a GFN denovo docking run get prepared and sanity-checked here, with **LRRK2 /
8TZC** as the worked example. This notebook is **diagnostic only — it changes no config files**;
it tells you what to put in `denovo_lrrk2.yml`, and Part 3 shows how.

1. **Protein / docking box.** Confirm the Uni-Dock search box (`denovo_lrrk2.yml`) actually sits on
   the binding site of the receptor that gets docked (`lrrk2.pdbqt`). The primary check needs only
   the **RCSB PDB + its co-crystal reference ligand** (MLI-2). A second structure from a different
   prep pipeline (BIOVIA `8TZC.pdb`) is *optional*: if you have one, the cross-frame cells diagnose a
   coordinate-frame mismatch; if you set `BIOVIA_PATH = None`, they auto-skip.
2. **Ligand seed.** This repo can start every trajectory from a fixed seed molecule and *freeze* part
   of it (`initial_scaffold` + `allowed_growth_atoms`/`frozen_atoms`). Those atoms are addressed by
   **0-based RDKit index**, so we draw the seed in 2D with each atom's index annotated — making it
   unambiguous which atoms you're freezing — then validate the choice with the repo's own builder.
3. **Config guide.** A markdown recipe translating your Part 1 box and Part 2 frozen-core choices into
   `denovo_lrrk2.yml`, with a paste-ready example and the launch command.

> `8TZC_RCSB.pdb` also carries **GDP** (in the GTPase / ROC domain, far from the kinase site) and a
> water; we filter HETATM to `A1N` so the reference ligand tracks the MLI-2 inhibitor, not GDP.

Run top-to-bottom. Requires `py3Dmol` and `rdkit` (both in the `agfn` env).
**Import order matters:** `py3Dmol` is imported *before* `rdkit.Chem.Draw` in the setup cell — the
reverse order segfaults the kernel. Keep it that way.

In [ ]:
# ----- Parameters (edit these) -----

# --- Part 1: protein / docking box ---
CONFIG_PATH     = "./src/config/denovo_lrrk2.yml"   # source of the configured docking box
DOCK_FRAME_PATH = "./data/docking/lrrk2.pdbqt"      # receptor actually used for docking (RCSB frame)
RCSB_PATH       = "./data/docking/8TZC_RCSB.pdb"    # official RCSB structure (same frame) + MLI-2 ligand
LIGAND_RESN     = "A1N"                             # co-crystal inhibitor MLI-2 (HETATM residue name)
# Optional second structure from a different prep pipeline, used ONLY for the cross-frame diagnostic.
# Set to None to skip that diagnostic and validate with the RCSB PDB + reference ligand alone.
BIOVIA_PATH     = "./data/docking/8TZC.pdb"         # BIOVIA / Discovery Studio prep (different frame); or None
VIEW_W, VIEW_H  = 620, 460

# --- Part 2: ligand seed / frozen-core ---
# The molecule you intend to seed `initial_scaffold` with. Defaults to the MLI-2 inhibitor (PDB
# chem-comp A1N) so this example is self-contained; if your config already sets
# initial_scaffold/seed_smiles, the cell below prefers that. Atoms are addressed by 0-based RDKit
# index (the order they appear here), exactly as `allowed_growth_atoms`/`frozen_atoms` expect.
SEED_SMILES = "COC1=CC=C(F)C(F)=C1C(C=C2)=CN3C2=NC(NC([C@@H]4[C@@H](C)C4)=O)=C3"  # MLI-2 (A1N)

# Preview a frozen/growth partition. Set EXACTLY ONE of these (the other stays None); leave both
# None to freeze the whole core while letting new atoms attach at any seed atom. Indices are the
# 0-based labels shown on the 2D drawing in Part 2.
ALLOWED_GROWTH_ATOMS = [22]    # e.g. [0, 3]  -> only these seed atoms may grow; all others frozen
FROZEN_ATOMS         = None    # e.g. [10, 11] -> these are frozen; every other seed atom may grow

In [ ]:
import sys
from pathlib import Path
import numpy as np

# The only boilerplate: make `import nbtools` resolve, then bootstrap the repo (chdir to the repo
# root so the configs' relative paths resolve, and put src/ on sys.path). Importing nbtools also
# imports py3Dmol first, so the py3Dmol-before-rdkit.Chem.Draw load order is handled for us.
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "nbtools").is_dir():
        sys.path.insert(0, str(_c)); break
    if (_c / "notebooks" / "nbtools").is_dir():
        sys.path.insert(0, str(_c / "notebooks")); break
import nbtools
from nbtools import pocket, render2d, render3d
REPO_ROOT = nbtools.setup_repo()
print("repo root:", REPO_ROOT)

# Part 1 — Protein / docking-box check

Is the configured Uni-Dock box centered on the binding site of the receptor we actually dock
(`lrrk2.pdbqt`)? The cells below read the box from the config exactly as the training backend does,
then check it two ways:

- **Primary (always runs):** does the box enclose the **reference co-crystal ligand** (MLI-2) in the
  RCSB structure, and does it sit on the receptor? Needs only `DOCK_FRAME_PATH` + `RCSB_PATH`.
- **Optional (needs `BIOVIA_PATH`):** if your box looks off, a second structure from a different prep
  pipeline lets us Kabsch-superpose the frames and detect a coordinate-frame mismatch. Auto-skips when
  `BIOVIA_PATH = None`.

In [ ]:
# Read the configured Uni-Dock box exactly the way the training backend does.
hps, center, size, receptor_cfg, target_name = pocket.read_box_from_config(CONFIG_PATH)
print(f"target             : {target_name}")
print(f"receptor (docking) : {receptor_cfg}")
print(f"box center         : {center}")
print(f"box size (A)       : {size}")
print(f"box min corner     : {np.round(center - size / 2, 2)}")
print(f"box max corner     : {np.round(center + size / 2, 2)}")

In [ ]:
# Primary check: does the configured box sit on the receptor and enclose the reference co-crystal
# ligand (MLI-2)? Everything here is in the docking (RCSB) frame.
res = pocket.primary_box_check(DOCK_FRAME_PATH, RCSB_PATH, LIGAND_RESN, center, size)
box_A = center                 # the configured box, as written in the config (what docking uses)
box_C = res["lig_center"]      # a box re-centered on the reference ligand
receptor_atoms = res["receptor"]

## Box overlays — does the box surround the binding site?

Magenta wireframe = the box from `denovo_lrrk2.yml`. A correctly configured box sits on the receptor
and wraps the green MLI-2 ligand sticks; a misconfigured one floats off in empty space.

In [ ]:
# Docking receptor (lrrk2.pdbqt) + configured box (magenta).
# A correct box sits on the protein; a frame-mismatched one floats off in empty space.
render3d.view_structure(DOCK_FRAME_PATH, "pdbqt", boxes=[(center, size, "magenta")],
                        w=VIEW_W, h=VIEW_H).show()

In [ ]:
# 8TZC_RCSB.pdb (same frame as the pdbqt) + configured box + MLI-2 (green sticks).
# The inhibitor marks the true pocket: a correct box should wrap the green sticks.
render3d.view_structure(RCSB_PATH, "pdb", boxes=[(center, size, "magenta")],
                        ligand_resn=LIGAND_RESN, w=VIEW_W, h=VIEW_H).show()

## Optional — cross-frame diagnostic (needs `BIOVIA_PATH`)

If the primary check flagged the box as off, the usual culprit is a **coordinate-frame mismatch**:
the box coordinates were picked in one structure's frame (e.g. a BIOVIA / Discovery Studio prep) but
the docking runs against a receptor in a different frame (RCSB). With a second structure of the same
protein we can superpose the two frames (Kabsch fit on matched atoms — they share residue numbering,
so no sequence alignment is needed) and map the configured box into the docking frame to confirm the
*intended* pocket and recover the corrected center.

**These cells auto-skip when `BIOVIA_PATH = None`.** They are a diagnostic, not part of the routine
check — a box that already passed the primary check needs none of this.

In [ ]:
# 8TZC.pdb (BIOVIA frame) + configured box. If the box was picked in this frame, it sits squarely
# on a pocket here even when it floats off the RCSB receptor above -- the tell-tale of a frame mismatch.
if BIOVIA_PATH and Path(BIOVIA_PATH).exists():
    render3d.view_structure(BIOVIA_PATH, "pdb", boxes=[(center, size, "magenta")],
                            w=VIEW_W, h=VIEW_H).show()
else:
    print("BIOVIA_PATH not set / not found -> skipping cross-frame diagnostic.")

Superpose `8TZC.pdb` (BIOVIA) onto `lrrk2.pdbqt` (RCSB) by matching atoms on
`(chain, resSeq, atom_name)`, then compare three same-size boxes against the MLI-2 pocket (box **C**,
the ground truth): **A** the configured center as-written, **B** that center reinterpreted as a
BIOVIA-frame point and mapped into the docking frame. Whichever of A or B lands on box C is the
center that's truly in the docking frame — if it's **A**, the box is already correct and needs no fix.

In [ ]:
# Superpose 8TZC.pdb (BIOVIA) onto lrrk2.pdbqt (RCSB) by matching atoms on (chain, resSeq, atom),
# then map the configured center into the docking frame (box B). Auto-skips without a BIOVIA frame.
box_B = None   # frame-corrected configured box; stays None when no BIOVIA frame is available
if BIOVIA_PATH and Path(BIOVIA_PATH).exists():
    box_B = pocket.crossframe_box(BIOVIA_PATH, DOCK_FRAME_PATH, center, size, receptor_atoms, box_C)
else:
    print("BIOVIA_PATH not set / not found -> skipping cross-frame superposition.")

In [ ]:
# Final superimpose on the RCSB structure (= docking frame, and it carries MLI-2):
#   red = box A (configured)   cyan = box B (frame-corrected)   orange = box C (MLI-2 centroid)
# Correct box: A and C coincide on the pocket and B flies off. Frame bug: A flies off, B lands on C.
if BIOVIA_PATH and Path(BIOVIA_PATH).exists() and box_B is not None:
    print("box colors:  red = configured (A)    cyan = frame-corrected (B)    "
          "orange = MLI-2 ligand box (C)    green sticks = MLI-2")
    render3d.view_structure(
        RCSB_PATH, "pdb",
        boxes=[(box_A, size, "red"), (box_B, size, "cyan"), (box_C, size, "orange")],
        ligand_resn=LIGAND_RESN, zoom_out=0.8, w=VIEW_W, h=VIEW_H).show()
else:
    print("BIOVIA_PATH not set / not found -> skipping 3-box comparison view.")

### Part 1 takeaway

- The **primary check** verdict is the one to trust: a good box encloses the reference ligand
  (`% enclosed` near 100) and is centered within a few Å of its centroid. If it says `CHECK`, fix the
  box — paste the ligand-centered center it prints into `target_grid.<target>` (Part 3), keeping
  `size` unchanged.
- The **cross-frame diagnostic** only matters when the primary check fails: it superposes a second
  structure and reports whether the configured center (box **A**) or its frame-corrected image (box
  **B**) lands on the MLI-2 pocket. If A is off and B lands on it, the coords were authored in the
  wrong frame — use box **B**'s center.
- Historical note for this example: the LRRK2 box was originally authored in the BIOVIA frame (~41 Å
  off the RCSB receptor) and has since been corrected in `denovo_lrrk2.yml`, so the primary check now
  passes and the diagnostic confirms box **A** is already in the docking frame. The cells stay
  verdict-driven so they remain useful for any target you point them at.

# Part 2 — Ligand seed / frozen-atom preparation

This repo can start **every** GFN trajectory from a fixed seed molecule and hold part of it immutable
(`initial_scaffold` in the config). The whole seed is a frozen *core* — its atoms, bonds, and atom
attributes never change — and you additionally choose **growth sites**: the subset of seed atoms at
which new atoms may attach. You declare them by atom index, two equivalent ways:

- `allowed_growth_atoms: [...]` — only these seed atoms may grow; every other seed atom is sealed.
- `frozen_atoms: [...]` — these atoms are sealed; every *other* seed atom may grow.

**The indices are 0-based RDKit parse order** — i.e. `atom.GetIdx()` on `Chem.MolFromSmiles(seed)`.
The repo's `build_frozen_seed_graph` asserts the seed's graph nodes are exactly `0..N-1` in that
order, so the numbers you read off the 2D drawing below are precisely the numbers the config consumes.
Hydrogens are implicit and are **not** indexed — you only ever address heavy atoms.

In [ ]:
# Resolve the seed molecule: prefer the config's initial_scaffold/seed_smiles if set, else the
# SEED_SMILES parameter (default MLI-2). Atoms are addressed by 0-based RDKit index.
from rdkit import Chem

cfg_seed = hps.get("initial_scaffold") or hps.get("seed_smiles")
seed_smiles = cfg_seed or SEED_SMILES
seed_src = "config (initial_scaffold/seed_smiles)" if cfg_seed else "SEED_SMILES parameter"
mol = Chem.MolFromSmiles(seed_smiles)
if mol is None:
    raise ValueError(f"seed SMILES did not parse: {seed_smiles!r}")
print(f"seed source : {seed_src}")
print(f"seed SMILES : {seed_smiles}")
print(f"heavy atoms : {mol.GetNumAtoms()}  (hydrogens implicit, not indexed)")
print()
print("index : symbol  (these are the numbers you put in allowed_growth_atoms / frozen_atoms)")
print("  " + ",  ".join(f"{a.GetIdx()}:{a.GetSymbol()}" for a in mol.GetAtoms()))

In [ ]:
# 2D depiction with each atom's 0-based index annotated -- read off which atoms to freeze / grow.
render2d.draw_mol_with_indices(mol, legend="seed atoms labelled by 0-based RDKit index",
                               size=(VIEW_W, VIEW_H))

In [ ]:
# Preview the partition implied by ALLOWED_GROWTH_ATOMS / FROZEN_ATOMS (set in the Parameters cell),
# computed exactly the way build_frozen_seed_graph does. Every seed atom is immutable; "growth
# sites" are the subset where new atoms may attach.
n = mol.GetNumAtoms()
growth = render2d.resolve_growth(n, ALLOWED_GROWTH_ATOMS, FROZEN_ATOMS)
sealed_only = set(range(n)) - growth
print(f"seed core (all atoms immutable): {n} atoms")
print(f"growth sites (green, new atoms may attach): {sorted(growth)}")
print(f"sealed-only  (red, no attachment)         : {sorted(sealed_only)}")
render2d.draw_mol_with_indices(
    mol, highlight=render2d.growth_partition_colors(n, growth),
    legend="green = growth site  |  red = sealed  (entire seed is an immutable core)",
    size=(VIEW_W, VIEW_H))

In [ ]:
# Authoritative check: build the seed graph with the repo's OWN builder, exactly as the trainer does
# (src/iterators/samp_iter_finetune.py). Validates atom types against the model's atom set, index
# ranges, and "at least one growth site", and prints the frozen_seed_size / frozen_growth_sites the
# training run will use. A red error here = a config that would fail at launch.
from gflownet.envs.mol_building_env import MolBuildingEnvContext, build_frozen_seed_graph

ctx = MolBuildingEnvContext(atoms=hps.atoms, max_nodes=hps.get("max_nodes"))
try:
    g = build_frozen_seed_graph(ctx, seed_smiles,
                                allowed_growth_atoms=ALLOWED_GROWTH_ATOMS, frozen_atoms=FROZEN_ATOMS)
    seed_n = g.graph["frozen_seed_size"]
    print("OK -- this spec is valid and trainable.")
    print(f"  frozen_seed_size    = {seed_n}")
    print(f"  frozen_growth_sites = {sorted(g.graph['frozen_growth_sites'])}")
    if ctx.max_nodes:
        print(f"  room to grow        = max_nodes({ctx.max_nodes}) - seed({seed_n}) "
              f"= {ctx.max_nodes - seed_n} atoms")
    print()
    print("Copy SEED_SMILES + your index list into denovo_lrrk2.yml as shown in Part 3.")
except Exception as e:
    print(f"INVALID -- {type(e).__name__}: {e}")
    print("Fix SEED_SMILES or the index list (Parameters cell) before training.")

# Part 3 — Configuring `denovo_lrrk2.yml`

Translate your Part 1 box and Part 2 seed choices into the config. **Everything lives under the
top-level `finetuning:` key** (loaded by `src/apps/docking/denovo/denovo_driver.py`). This notebook
writes nothing — edit the YAML yourself.

### Docking box (Part 1)
Select the target with `target_name`, and define its box under `target_grid.<target_name>`:

| field | meaning |
|---|---|
| `receptor` | path to the `.pdbqt` that gets docked (`./data/docking/lrrk2.pdbqt`) |
| `center_x/y/z` | box center, **Å, in the receptor's own coordinate frame** |
| `size_x/y/z` | box edge lengths, Å |

If Part 1's primary check said `CHECK`, paste the **ligand-centered center** it printed (or box **B**
from the cross-frame diagnostic) into `center_x/y/z`, leaving `size_*` unchanged.

### Seed + frozen atoms (Part 2)
Set `initial_scaffold` to your seed SMILES, then choose **exactly one** growth spec (or neither):

| you set | effect |
|---|---|
| `allowed_growth_atoms: [i, j, …]` | only atoms *i, j, …* may grow; all other seed atoms sealed |
| `frozen_atoms: [i, j, …]` | atoms *i, j, …* sealed; every other seed atom may grow |
| neither | whole core frozen, but new atoms may attach at **any** seed atom |

Indices are the **0-based numbers from the Part 2 drawing**. Setting both raises an error; sealing
every atom raises an error.

**Gotchas (all checked by the Part 2 validation cell):**
- When `initial_scaffold` is set, `seed_smiles` / `seed_scaffold` are **ignored**.
- Frozen-core runs **online-only** — the trainer auto-forces `offline_data: False` and prints
  `[frozen-core] forcing ONLINE-ONLY mode` at startup (offline dataset molecules can't preserve the core).
- Seed atoms must be within the model's atom set `atoms: ['Br','C','Cl','F','I','N','O','S']`.
- Seed size must be `< max_nodes` (here 45) to leave room to grow.

### Paste-ready example (LRRK2 + MLI-2 seed)
```yaml
finetuning:
  target_name: "lrrk2"
  target_grid:
    lrrk2:
      receptor: "./data/docking/lrrk2.pdbqt"
      center_x: 185.7        # corrected pocket center (Part 1); was a BIOVIA-frame value
      center_y: 178.1
      center_z: 144.7
      size_x: 22.032
      size_y: 22.032
      size_z: 22.032

  # Frozen-core seed: start from MLI-2 and grow only at the atoms you chose in Part 2.
  initial_scaffold: "C[C@H]1CN(C[C@@H](C)O1)c2cc(ncn2)c3n[nH]c4ccc(OC5(C)CC5)cc34"
  allowed_growth_atoms: [16]   # <-- replace with the green indices from your Part 2 drawing
  frozen_atoms: null           # leave null when using allowed_growth_atoms
  # offline_data is forced to False automatically when initial_scaffold is set
```

### Launch
```bash
conda run -n agfn python ./src/apps/docking/denovo/denovo_driver.py ./src/config/denovo_lrrk2.yml
```